# Paper-exact reproduction: multilingual circuits on Qwen3-4B

Reproduces the biology paper's Multilingual Circuits dive
([biology.html §Multilingual](https://transformer-circuits.pub/2025/attribution-graphs/biology.html#dives-multilingual))
on Qwen3-4B + `mwhanna/qwen3-4b-transcoders`. Paper values: `notes/biology_digest.md` §B;
verdicts: the Summary below and `DEVLOG.md`.

**Setup facts**
- Both prompt formats run side by side: the paper's exact **raw** open-quote completions
  (`The opposite of "small" is "`, one sink token prepended, `n_bos_tokens=1`) and our
  **chat** instruction adaptation (Qwen3-4B is an instruct model; the answer is the first
  generated token). The **language swap is raw-only** — its quote supernodes exist only on
  the raw pages: chat prompts end on assistant-header scaffold tokens, not the
  content-bearing open quote the paper's swap intervenes on.
- Steering: paper multiples are multiplicative M, ours additive `m = M − 1`
  (paper −5× ⇒ m=−6, −0.5× ⇒ m=−1.5). Donors are injected at the absolute
  `value = mult × stored act` recorded on their donor graph.

**v2 pipeline** — interventions run the paper's **constrained patching** only
(activations at layers ≤ ℓ pinned at their perturbed values, the real model after ℓ;
ℓ swept per experiment on p(expected) and stated per result). Supernodes load from the
**reviewed artifact** `supernodes/multilingual_4b.json` — hand-selected in the explorer
review pages ("Export groups"), ingested by `build_supernodes.py --from-exports`,
user-reviewed 2026-07-11 — and graphs load from the `build_supernode_inputs.py` dumps
that review saw. v1 (propagate mode, in-notebook influence/activation selection) lives
in git history and `DEVLOG.md`.

## What actually runs — exact model inputs, supernodes, artifacts

### The ten graphs (loaded from `artifacts/supernode_inputs/4b/`, manifest-checked)

| graph | format | prompt (chat = the user line of the templated input; raw = sink token prepended) | answer | tokens | final | operand | prune |
|---|---|---|---|---|---|---|---|
| `antonym_en` | chat | `What is the opposite of "small"? Reply with only the word, nothing else.` | 'large' | 29 | 28 | 9 | 0.8 |
| `antonym_fr` | chat | `Quel est le contraire de "petit" ? Réponds avec un seul mot, rien d'autre.` | 'Grand' | 37 | 36 | 12 | 0.8 |
| `antonym_zh` | chat | `"小"的反义词是什么？只回答一个词，不要其他内容。` | '大' | 30 | 29 | 4 | 0.8 |
| `synonym_en` | chat | `Give one synonym of "small". Reply with only the word, nothing else.` | 'tiny' | 28 | 27 | 8 | 0.8 |
| `hot_en` | chat | `What is the opposite of "hot"? Reply with only the word, nothing else.` | 'cold' | 29 | 28 | 9 | 0.8 |
| `raw_antonym_en` | raw | `The opposite of "small" is "` | 'large' | 9 | 8 | 5 | 0.95 |
| `raw_antonym_fr` | raw | `Le contraire de "petit" est "` | 'grand' | 11 | 10 | 7 | 0.95 |
| `raw_antonym_zh` | raw | `"小"的反义词是"` | '大' | 10 | 9 | 2 | 0.95 |
| `raw_synonym_en` | raw | `A synonym of "small" is "` | 'tiny' | 9 | 8 | 5 | 0.95 |
| `raw_hot_en` | raw | `The opposite of "hot" is "` | 'cold' | 9 | 8 | 5 | 0.95 |

Chat prompts go through the Qwen3 chat template with thinking disabled — the final
position is the last assistant-header scaffold token, and the answer is the first
generated token. Raw prompts are the paper's exact completions with one sink token
prepended (transcoders cannot reconstruct position 0); their final position IS the
content-bearing open quote. Chat graphs are pruned at 0.8 (circuit-tracer default),
raw at 0.95 (at 0.8 they starve — no quote-position nodes survive for the detectors).

### The reviewed supernodes (`supernodes/multilingual_4b.json`, 16, `approved: true`)

| supernode | role | members | layers | reviewed on (per-graph acts) |
|---|---|---|---|---|
| `antonym (multilingual)` | source | 7 | L7–L34 | antonym_en, antonym_fr, antonym_zh, raw_antonym_en, raw_antonym_fr, raw_antonym_zh |
| `small (multilingual)` | source | 6 | L27–L32 | antonym_en, antonym_fr, antonym_zh, raw_antonym_en, raw_antonym_fr, raw_antonym_zh |
| `say large (multilingual)` | readout | 7 | L30–L32 | antonym_en, antonym_fr, antonym_zh, raw_antonym_en, raw_antonym_fr, raw_antonym_zh |
| `say large (en)` | readout | 3 | L23–L33 | antonym_en |
| `opposite (en)` | input | 6 | L23–L28 | antonym_en, raw_antonym_en |
| `say large (fr)` | readout | 6 | L30–L33 | antonym_fr, raw_antonym_fr |
| `opposite (fr)` | input | 6 | L23–L31 | antonym_fr, raw_antonym_fr |
| `say large (zh)` | readout | 2 | L25–L27 | antonym_zh |
| `opposite (zh)` | input | 6 | L15–L34 | antonym_zh, raw_antonym_zh |
| `hot (multilingual)` | donor | 6 | L24–L32 | hot_en, raw_hot_en |
| `say cold (multilingual)` | readout | 6 | L27–L32 | hot_en, raw_hot_en |
| `quote (en)` | source+donor | 6 | L23–L29 | raw_antonym_en |
| `quote (fr)` | source+donor | 6 | L29–L32 | raw_antonym_fr |
| `quote (zh)` | source+donor | 6 | L30–L34 | raw_antonym_zh |
| `synonym (multilingual)` | donor | 8 | L7–L34 | raw_synonym_en, synonym_en |
| `say small (multilingual)` | readout | 7 | L28–L33 | raw_synonym_en, synonym_en |

Unions span chat + raw pages (the group NAME merged identically-named groups at
ingestion); every member carries per-graph `acts` and `positions`. Supernodes are
globally disjoint by (layer, feature) — the loader enforces it. Admitted absent after
review: `large (multilingual)`, `cold (multilingual)`, `say hot`.

### The three swaps (Fig B2; all constrained-only, ℓ swept per job)

| § | swap | format | source (steer at member node positions) | donor (inject `value = mult × stored act`) | readouts (rows ≤ ℓ pinned) |
|---|---|---|---|---|---|
| C | operation: antonym → synonym (Fig B3) | chat + raw | `antonym (multilingual)` −5× (m=−6); final-position fallback | `synonym (multilingual)` +6× at the recipient's operation-word span (opposite/contraire/反义词) | say small ↑ (vs donor-graph act), say large ↓, opposite ({lg}), antonym response |
| D | operand: small → hot (Fig B4) | chat + raw | `small (multilingual)` −0.5× (m=−1.5); operand-position fallback | `hot (multilingual)` +1.5× at the recipient's operand token | say cold ↑ (vs donor-graph act), say large ↓, opposite ({lg}), small response |
| E | language: X → Y (Fig B5) | **raw only** | `quote (X)` −5× (m=−6) at the final open quote | `quote (Y)` +6× at the final open quote | say large (multilingual) ≈, say large (X) ↓, say large (Y) ↑, quote responses |

### Artifacts (`artifacts/paper_multilingual/4b/`)

`behavior.json` / `raw_behavior.json` (§0); per swap `{operation,operand,language}_results_constrained.json`
(sweep curves + the ℓ decision curve per job), `*_readouts_constrained.json`,
`*_sweep_constrained.png`; `language_ladder_readouts_constrained.json` (§E's 1×/3×/6×
per-feature ladder); `overlap_curves.json` + `overlap_paper.png` (§F);
`graph_<name>.html` explorers with the reviewed supernodes pre-loaded as groups;
`../8b/overlap_curves.json` + `overlap_scale_comparison.png` (§H).

In [ ]:
%matplotlib inline
import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import multilingual_helper as M
import numpy as np
import torch
from build_supernodes import op_word_positions

from llm_circuits.models.qwen3 import load_qwen3
from llm_circuits.settings import artifacts_dir, default_device
from llm_circuits.transcoders.circuit_tracer_loader import load_transcoder

SIZE = "4b"
DEVICE = default_device()
OUT = artifacts_dir() / "paper_multilingual" / SIZE
OUT.mkdir(parents=True, exist_ok=True)
SN_INPUTS = artifacts_dir() / "supernode_inputs" / SIZE

model, tokenizer = load_qwen3(SIZE, dtype_str="bf16", device_map=DEVICE)
model.eval()
# Eager decoders (lazy_decoder=False): interventions decode every layer repeatedly, and
# lazy mode re-streams ~28 GB of W_dec per clean forward — the notebook's dominant cost.
# Eager fits an A100-80GB for the 4b (CLAUDE.md perf notes); the 8b scale section (§H)
# keeps lazy decoders — its overlap analysis is encode-only.
tc = load_transcoder(
    f"qwen3-{SIZE}", device=DEVICE, dtype=torch.bfloat16, lazy_decoder=False
).transcoder
N_LAYERS = len(tc)
print(f"Qwen3-{SIZE} on {DEVICE}; {N_LAYERS} transcoder layers; artifacts -> {OUT}")

## 0. Behavior: the antonym task in EN/FR/ZH (+ swap targets), both formats

Paper behavior (raw prompts): big/large / grand / 大. Greedy generation on the nine chat
prompts and nine raw completions records each answer's **first token id** — these become
`baseline_token` / `expected_token` for every swap (baseline = the antonym answer;
expected = the synonym answer for §C, the hot-antonym answer for §D, the other
language's antonym answer for §E). The antonym answers must match the graph dumps'
manifest — asserted in §A. Known Qwen3-4B degeneracies (both formats, so not chat
artifacts): the FR synonym echoes the operand (petit→petit), the ZH synonym sits on a
小/微 near-tie — "the synonym answer" for §C is whatever the model itself does.

In [ ]:
behavior = {"antonym": {}, "synonym": {}, "antonym_hot": {}}
raw_behavior = {"antonym": {}, "synonym": {}, "antonym_hot": {}}
for lg in M.LANGS:
    for key, prompt in [
        ("antonym", M.antonym_prompt("small", lg)),
        ("synonym", M.synonym_prompt("small", lg)),
        ("antonym_hot", M.antonym_prompt("hot", lg)),
    ]:
        tid, tstr, cont = M.model_answer(model, tokenizer, prompt)
        behavior[key][lg] = {"token_id": tid, "token": tstr, "continuation": cont}
        print(f"chat {key:12s} {lg}: {tstr!r}  (full: {cont!r})")
    for key, prompt in [
        ("antonym", M.raw_antonym_prompt("small", lg)),
        ("synonym", M.raw_synonym_prompt("small", lg)),
        ("antonym_hot", M.raw_antonym_prompt("hot", lg)),
    ]:
        tid, tstr, cont = M.model_answer(model, tokenizer, prompt, raw=True)
        raw_behavior[key][lg] = {"token_id": tid, "token": tstr, "continuation": cont}
        print(f"raw  {key:12s} {lg}: {tstr!r}  (full: {cont!r})")
(OUT / "behavior.json").write_text(json.dumps(behavior, indent=1, ensure_ascii=False))
(OUT / "raw_behavior.json").write_text(json.dumps(raw_behavior, indent=1, ensure_ascii=False))

## A. Attribution graphs — loaded from the reviewed dumps

The ten pruned graphs (3 antonym recipients + EN synonym/hot donors, chat + raw) load
from `artifacts/supernode_inputs/4b/` — the exact dumps the supernode review pages were
built from (`build_supernode_inputs.py`; bf16 rebuilds are bit-stable across A100s).
Prompt tokenization is re-derived and length-checked against the manifest so every
intervention position below lands where the review looked.

In [ ]:
manifest = json.loads((SN_INPUTS / "manifest.json").read_text())
GRAPH_NAMES = [
    "antonym_en",
    "antonym_fr",
    "antonym_zh",
    "synonym_en",
    "hot_en",
    "raw_antonym_en",
    "raw_antonym_fr",
    "raw_antonym_zh",
    "raw_synonym_en",
    "raw_hot_en",
]
dev = next(model.parameters()).device
graphs, ids, fin = {}, {}, {}
for name in GRAPH_NAMES:
    info = manifest["graphs"][name]
    graphs[name] = json.loads((SN_INPUTS / f"graph_{name}.json").read_text())
    tok_fn = M.tokenize_raw if info["raw"] else M.tokenize
    ids[name] = tok_fn(tokenizer, info["prompt"], dev)
    assert ids[name].shape[1] == info["n_tokens"], name
    fin[name] = info["final_position"]
    nfeat = sum(1 for n in graphs[name]["nodes"] if n["node_type"] == "feature")
    print(
        f"{name}: {nfeat} feature nodes (prune {info['node_threshold']}, git "
        f"{manifest['git']}), answer {info['answer']!r}"
    )
# the behavior cell's antonym answers must be the dumps' answers
for lg in M.LANGS:
    assert (
        behavior["antonym"][lg]["token_id"]
        == manifest["graphs"][f"antonym_{lg}"]["answer_token_id"]
    )
    assert (
        raw_behavior["antonym"][lg]["token_id"]
        == manifest["graphs"][f"raw_antonym_{lg}"]["answer_token_id"]
    )
print("behavior == manifest answers for all six antonym graphs")

## B. Supernodes — the reviewed artifact (Fig B1's node classes)

All selection lives in `supernodes/multilingual_4b.json`: seeds proposed by the semantic
scan (`build_supernodes.py`), groups adjusted and approved by hand on the explorer
review pages, "Export groups" JSONs ingested verbatim (`--from-exports`; the exports are
committed under `supernodes/exports/` as the review record). Identically-named groups
across pages merged into one supernode with per-graph `acts`/`positions` per member.
The loader refuses unapproved files, rejected members, and any (layer, feature) claimed
by two supernodes. The full membership table is in the at-a-glance cell above.

In [ ]:
SN = M.load_supernodes(f"supernodes/multilingual_{SIZE}.json")
_doc = json.loads(Path(f"supernodes/multilingual_{SIZE}.json").read_text())
print(_doc.get("review_log", ""), "\n")
for name, sn in SN.items():
    span = sorted({m["layer"] for m in sn["members"]})
    on = sorted({g for m in sn["members"] for g in (m.get("acts") or {})})
    print(
        f"{name:26s} [{sn['role']:12s}] n={len(sn['members'])}  L{span[0]}-L{span[-1]}  on: {', '.join(on)}"
    )

In [ ]:
# Fig B1's shared multilingual core, on the loaded dumps + the reviewed supernodes.
feat_sets = {
    name: {
        (n["layer"], n["feature_idx"]) for n in graphs[name]["nodes"] if n["node_type"] == "feature"
    }
    for name in GRAPH_NAMES
}
for fmt in ("chat", "raw"):
    names = {lg: (f"antonym_{lg}" if fmt == "chat" else f"raw_antonym_{lg}") for lg in M.LANGS}
    inter3 = set.intersection(*(feat_sets[n] for n in names.values()))
    sizes = ", ".join(f"{lg} {len(feat_sets[n])}" for lg, n in names.items())
    print(f"{fmt} antonym graphs: {sizes}; features in all three: {len(inter3)}")
    for a, b in (("en", "fr"), ("en", "zh"), ("fr", "zh")):
        print(f"   {a} & {b}: {len(feat_sets[names[a]] & feat_sets[names[b]])}")

# the paper's 20/27 analogue: members of the cross-language supernodes present on all
# three antonym pages (per the reviewed per-graph acts — presence IS the review record)
print()
for sn_name in ("antonym (multilingual)", "small (multilingual)", "say large (multilingual)"):
    sn = SN[sn_name]
    chat3 = sum(1 for m in sn["members"] if all(f"antonym_{lg}" in m["acts"] for lg in M.LANGS))
    raw3 = sum(1 for m in sn["members"] if all(f"raw_antonym_{lg}" in m["acts"] for lg in M.LANGS))
    print(
        f"{sn_name}: {len(sn['members'])} members — on all three chat pages: {chat3}; all three raw: {raw3}"
    )
print()
for m in SN["say large (multilingual)"]["members"]:
    tops = m["evidence"]["top_logits"][:4]
    print(f"  say-large L{m['layer']}f{m['feature']}: on {sorted(m['acts'])}; top logits {tops}")

## C. Operation swap: antonym → synonym (Fig B3; −5× source, +6× donor)

Per recipient (3 languages × chat/raw): every `antonym (multilingual)` member is steered
to −5× its clean activation (m=−6) **at its own node position in the recipient graph**
(members the review only saw elsewhere fall back to the final position — the m
convention makes that a no-op where the feature is inactive, but the member still sets
the sweep floor `l_max`); the `synonym (multilingual)` donors are injected at
`value = +6× their stored act` on the format's donor graph (`synonym_en` /
`raw_synonym_en` — members without an act there sit out), at the recipient's
operation-word token span (`opposite`/`contraire`/`反义词`). Paper: language-appropriate
synonyms top-1, crossover ≈4× in every language. Track the actual top tokens — Qwen3's
own synonym behavior (FR echo, ZH 小/微 tie) is the honest target.

**Protocol note (measured, not chosen):** the reviewed antonym supernode carries an L34
member, so `l_max = 34` in every job — the ℓ sweep has two points (34, 35) and the
paper's protocol here degenerates to (nearly) the steered features' direct logit
effect. All Fig B3 %-readout rows sit at L33 or below ⇒ **pinned** by the protocol
(reported as `n_pinned`; the sweeps and crossovers carry the result). This is the same
selection-style × protocol interaction the propagate-era run documented — now measured
on the reviewed selection.

In [ ]:
kind = "operation"
_, don_max = M.PAPER_SWAP_STRENGTHS[kind]
op_results, op_readouts = {}, {}
for fmt in ("chat", "raw"):
    beh = behavior if fmt == "chat" else raw_behavior
    donor_graph = "synonym_en" if fmt == "chat" else "raw_synonym_en"
    for lg in M.LANGS:
        rec = f"antonym_{lg}" if fmt == "chat" else f"raw_antonym_{lg}"
        span = sorted(op_word_positions(graphs[rec], M.OP_WORD[lg]))
        assert span, f"operation word not found in {rec}"
        ivs_fn = M.swap_ivs_fn(
            SN["antonym (multilingual)"],
            rec,
            fin[rec],
            SN["synonym (multilingual)"],
            donor_graph,
            span,
            kind=kind,
        )
        expected = beh["synonym"][lg]["token_id"]
        ell, curve = M.choose_swap_end_layer(model, tc, ids[rec], ivs_fn(don_max), expected)
        r = M.supernode_swap_sweep(
            model,
            tc,
            ids[rec],
            ivs_fn,
            tokenizer,
            kind=kind,
            baseline_token=beh["antonym"][lg]["token_id"],
            expected_token=expected,
            patch_end_layer=ell,
        )
        r["l_max"] = curve.end_layers[0]
        r["ell_curve"] = {
            "end_layers": curve.end_layers,
            "p_expected": [round(p, 4) for p in curve.probs],
        }
        r["donor_span"] = {int(p): graphs[rec]["tokens"][p] for p in span}
        op_results[f"{fmt}_{lg}"] = r
        print(
            f"{fmt} {lg}: l_max={r['l_max']} ell={ell} | crossover={r['crossover']} "
            f"p_exp(max)={max(r['p_expected']):.3f} endpoint_top={r['top_tokens'][-1][:3]}"
        )
        # Fig B3 %-readouts at the endpoint (rows <= ell pinned by the protocol)
        ro = {
            "say_small_pct_stored": (SN["say small (multilingual)"], donor_graph),
            "say_large_pct_baseline": (SN["say large (multilingual)"], None),
            "opposite_lang_pct_baseline": (SN[f"opposite ({lg})"], None),
            "antonym_response_pct_baseline": (SN["antonym (multilingual)"], None),
        }
        rl = M.readout_layers_above([sn for sn, _ in ro.values()], ell)
        res = M.run_feature_intervention(
            model,
            tc,
            ids[rec],
            ivs_fn(don_max),
            n_bos_tokens=M.N_BOS,
            patch_end_layer=ell,
            readout_layers=rl,
        )
        row = {
            name: M.supernode_readout(
                res, sn, rec, final_pos=fin[rec], ref_graph=rg, patch_end_layer=ell
            )
            for name, (sn, rg) in ro.items()
        }
        op_readouts[f"{fmt}_{lg}@{don_max:g}x"] = row
        M.print_readout_row(f"{fmt} {lg} @ {don_max:g}x [ell={ell}]", row)

fig, axes = plt.subplots(2, 3, figsize=(12.8, 7.0))
for ri, fmt in enumerate(("chat", "raw")):
    beh = behavior if fmt == "chat" else raw_behavior
    token_strs = {
        lg: (
            tokenizer.decode([beh["antonym"][lg]["token_id"]]).strip(),
            tokenizer.decode([beh["synonym"][lg]["token_id"]]).strip(),
        )
        for lg in M.LANGS
    }
    M.plot_swap_sweeps(
        {lg: op_results[f"{fmt}_{lg}"] for lg in M.LANGS},
        tokenizer,
        token_strs,
        title="",
        ax_row=axes[ri],
    )
    axes[ri][0].set_ylabel(f"{fmt}\nnext-token probability")
fig.suptitle("operation swap antonym->synonym (constrained patching at the swept ell)")
fig.tight_layout()
fig.savefig(OUT / "operation_sweep_constrained.png", dpi=130, bbox_inches="tight")
plt.show()
(OUT / "operation_results_constrained.json").write_text(
    json.dumps(op_results, indent=1, ensure_ascii=False, default=str)
)
(OUT / "operation_readouts_constrained.json").write_text(
    json.dumps(op_readouts, indent=1, ensure_ascii=False, default=str)
)

## D. Operand swap: small → hot (Fig B4; −0.5× source, +1.5× donor)

Per recipient (3 languages × chat/raw): `small (multilingual)` steered to −0.5× (m=−1.5)
at each member's operand-token position (fallback: the manifest's operand position);
`hot (multilingual)` injected at `value = +1.5× stored act` on the format's donor graph
(`hot_en` / `raw_hot_en`) at the recipient's operand token. Readout at the final
position. Paper: cold / f[roid] / 冷 — its cleanest intervention. Expected tokens are
the model's own hot-antonym answers per format (§0).

**Protocol note:** `small (multilingual)` reaches L32 and every say-cold/say-large
readout member sits at L32 or below, so with ℓ ≥ l_max = 32 the Fig B4 %-readouts are
pinned at every admissible ℓ — under the paper's protocol at this reviewed selection,
"say cold recruited" is unmeasurable and the sweeps carry the verdict (the propagate-era
run, which could read it, measured 65–148% recruitment; git history / DEVLOG).

In [ ]:
kind = "operand"
_, don_max = M.PAPER_SWAP_STRENGTHS[kind]
od_results, od_readouts = {}, {}
for fmt in ("chat", "raw"):
    beh = behavior if fmt == "chat" else raw_behavior
    donor_graph = "hot_en" if fmt == "chat" else "raw_hot_en"
    for lg in M.LANGS:
        rec = f"antonym_{lg}" if fmt == "chat" else f"raw_antonym_{lg}"
        opos = manifest["graphs"][rec]["operand_position"]
        ivs_fn = M.swap_ivs_fn(
            SN["small (multilingual)"],
            rec,
            opos,
            SN["hot (multilingual)"],
            donor_graph,
            [opos],
            kind=kind,
        )
        expected = beh["antonym_hot"][lg]["token_id"]
        ell, curve = M.choose_swap_end_layer(model, tc, ids[rec], ivs_fn(don_max), expected)
        r = M.supernode_swap_sweep(
            model,
            tc,
            ids[rec],
            ivs_fn,
            tokenizer,
            kind=kind,
            baseline_token=beh["antonym"][lg]["token_id"],
            expected_token=expected,
            patch_end_layer=ell,
        )
        r["l_max"] = curve.end_layers[0]
        r["ell_curve"] = {
            "end_layers": curve.end_layers,
            "p_expected": [round(p, 4) for p in curve.probs],
        }
        r["operand_position"] = {int(opos): graphs[rec]["tokens"][opos]}
        od_results[f"{fmt}_{lg}"] = r
        print(
            f"{fmt} {lg}: l_max={r['l_max']} ell={ell} | crossover={r['crossover']} "
            f"p_exp(max)={max(r['p_expected']):.3f} endpoint_top={r['top_tokens'][-1][:3]}"
        )
        ro = {
            "say_cold_pct_stored": (SN["say cold (multilingual)"], donor_graph),
            "say_large_pct_baseline": (SN["say large (multilingual)"], None),
            "opposite_lang_pct_baseline": (SN[f"opposite ({lg})"], None),
            "small_response_pct_baseline": (SN["small (multilingual)"], None),
        }
        rl = M.readout_layers_above([sn for sn, _ in ro.values()], ell)
        res = M.run_feature_intervention(
            model,
            tc,
            ids[rec],
            ivs_fn(don_max),
            n_bos_tokens=M.N_BOS,
            patch_end_layer=ell,
            readout_layers=rl,
        )
        row = {
            name: M.supernode_readout(
                res, sn, rec, final_pos=fin[rec], ref_graph=rg, patch_end_layer=ell
            )
            for name, (sn, rg) in ro.items()
        }
        od_readouts[f"{fmt}_{lg}@{don_max:g}x"] = row
        M.print_readout_row(f"{fmt} {lg} @ {don_max:g}x [ell={ell}]", row)

fig, axes = plt.subplots(2, 3, figsize=(12.8, 7.0))
for ri, fmt in enumerate(("chat", "raw")):
    beh = behavior if fmt == "chat" else raw_behavior
    token_strs = {
        lg: (
            tokenizer.decode([beh["antonym"][lg]["token_id"]]).strip(),
            tokenizer.decode([beh["antonym_hot"][lg]["token_id"]]).strip(),
        )
        for lg in M.LANGS
    }
    M.plot_swap_sweeps(
        {lg: od_results[f"{fmt}_{lg}"] for lg in M.LANGS},
        tokenizer,
        token_strs,
        title="",
        ax_row=axes[ri],
    )
    axes[ri][0].set_ylabel(f"{fmt}\nnext-token probability")
fig.suptitle("operand swap small->hot (constrained patching at the swept ell)")
fig.tight_layout()
fig.savefig(OUT / "operand_sweep_constrained.png", dpi=130, bbox_inches="tight")
plt.show()
(OUT / "operand_results_constrained.json").write_text(
    json.dumps(od_results, indent=1, ensure_ascii=False, default=str)
)
(OUT / "operand_readouts_constrained.json").write_text(
    json.dumps(od_readouts, indent=1, ensure_ascii=False, default=str)
)

## E. Language swap: X → Y (Fig B5; −5× source, +6× donor) — raw pages only

Paper protocol: swap the language-tracking features on the final token — `quote (X)`
steered to −5× (m=−6), `quote (Y)` injected at `value = +6× its stored act` at the
recipient's final open-quote position — EN→ZH, FR→EN, ZH→FR; the output language should
change while operation and operand are preserved (say-large-multilingual ≈100%, old
say-large-X 18–39%, new say-large-Y 76–105% in the paper's annotations). **Raw-only by
construction:** the quote supernodes exist only on the raw pages — the chat prompts'
final tokens are assistant-header scaffold, and the review found no defensible chat
quote nodes. Qwen3 caveat measured at selection time: the reviewed quote features sit at
L23–L34, not the paper's early layers — Qwen3's raw graphs keep no early
quote-position nodes at all, so the paper's "early detection" handle does not exist
here and the sweep floor `l_max` is high (32–34) in every direction.

`say large (X)` readouts: the source language's reads against its recipient baseline;
the target language's against its stored act (`say large (zh)` was reviewed only on the
chat page, so its stored reference is the chat act — stated where it matters).

In [ ]:
kind = "language"
_, don_max = M.PAPER_SWAP_STRENGTHS[kind]
DIRECTIONS = [("en", "zh"), ("fr", "en"), ("zh", "fr")]
lang_results, lang_readouts, lang_ivs_fns, lang_ells = {}, {}, {}, {}
for src_lg, tgt_lg in DIRECTIONS:
    key = f"{src_lg}->{tgt_lg}"
    rec = f"raw_antonym_{src_lg}"
    donor_graph = f"raw_antonym_{tgt_lg}"
    ivs_fn = M.swap_ivs_fn(
        SN[f"quote ({src_lg})"],
        rec,
        fin[rec],
        SN[f"quote ({tgt_lg})"],
        donor_graph,
        [fin[rec]],
        kind=kind,
    )
    lang_ivs_fns[key] = ivs_fn
    expected = raw_behavior["antonym"][tgt_lg]["token_id"]
    ell, curve = M.choose_swap_end_layer(model, tc, ids[rec], ivs_fn(don_max), expected)
    lang_ells[key] = ell
    vac = " -- p_exp ~0 at EVERY ell (no landing layer exists)" if max(curve.probs) < 1e-3 else ""
    r = M.supernode_swap_sweep(
        model,
        tc,
        ids[rec],
        ivs_fn,
        tokenizer,
        kind=kind,
        baseline_token=raw_behavior["antonym"][src_lg]["token_id"],
        expected_token=expected,
        patch_end_layer=ell,
    )
    r["l_max"] = curve.end_layers[0]
    r["ell_curve"] = {
        "end_layers": curve.end_layers,
        "p_expected": [round(p, 6) for p in curve.probs],
    }
    lang_results[key] = r
    print(
        f"{key}: l_max={r['l_max']} ell={ell} p_exp max over ell {max(curve.probs):.4f}{vac} | "
        f"crossover={r['crossover']} endpoint_top={r['top_tokens'][-1][:3]}"
    )
    ro = {
        "say_large_multi_pct_baseline": (SN["say large (multilingual)"], None),
        "say_large_src_pct_baseline": (SN[f"say large ({src_lg})"], None),
        "say_large_tgt_pct_stored": (SN[f"say large ({tgt_lg})"], donor_graph),
        "quote_src_response_pct_baseline": (SN[f"quote ({src_lg})"], None),
        "quote_tgt_response_pct_stored": (SN[f"quote ({tgt_lg})"], donor_graph),
    }
    rl = M.readout_layers_above([sn for sn, _ in ro.values()], ell)
    res = M.run_feature_intervention(
        model,
        tc,
        ids[rec],
        ivs_fn(don_max),
        n_bos_tokens=M.N_BOS,
        patch_end_layer=ell,
        readout_layers=rl,
    )
    row = {
        name: M.supernode_readout(
            res, sn, rec, final_pos=fin[rec], ref_graph=rg, patch_end_layer=ell
        )
        for name, (sn, rg) in ro.items()
    }
    lang_readouts[f"{key}@{don_max:g}x"] = row
    M.print_readout_row(f"{key} @ {don_max:g}x [ell={ell}]", row)

token_strs = {
    f"{a}->{b}": (
        tokenizer.decode([raw_behavior["antonym"][a]["token_id"]]).strip(),
        tokenizer.decode([raw_behavior["antonym"][b]["token_id"]]).strip(),
    )
    for a, b in DIRECTIONS
}
ax_row = M.plot_swap_sweeps(lang_results, tokenizer, token_strs, title="")
plt.suptitle("language swap on the raw open-quote prompts (constrained patching)")
plt.tight_layout()
plt.savefig(OUT / "language_sweep_constrained.png", dpi=130, bbox_inches="tight")
plt.show()
(OUT / "language_results_constrained.json").write_text(
    json.dumps(lang_results, indent=1, ensure_ascii=False, default=str)
)
(OUT / "language_readouts_constrained.json").write_text(
    json.dumps(lang_readouts, indent=1, ensure_ascii=False, default=str)
)

In [ ]:
# Fig B5-style per-feature ladder for raw en->zh at 1x/3x/6x (the F8 recheck): does the
# zh say-stage EVER come online while the en quote supernode is driven at paper
# strengths? Constrained at the direction's chosen ell; with the reviewed quote features
# at L23-L34, every say-large row below ell reports as pinned — the protocol-honest
# answer (the propagate-era diagnostic that could read them is in git history/DEVLOG).
ivs_fn = lang_ivs_fns["en->zh"]
ell = lang_ells["en->zh"]
rec = "raw_antonym_en"
groups = {
    "say_large_multi": (SN["say large (multilingual)"], None),
    "say_large_en": (SN["say large (en)"], None),
    "say_large_zh": (SN["say large (zh)"], "raw_antonym_zh"),
}
rl = M.readout_layers_above([sn for sn, _ in groups.values()], ell)
ladder = {}
for s in (1.0, 3.0, 6.0):
    res = M.run_feature_intervention(
        model,
        tc,
        ids[rec],
        ivs_fn(s),
        n_bos_tokens=M.N_BOS,
        patch_end_layer=ell,
        readout_layers=rl,
    )
    row = {
        name: M.supernode_readout(
            res, sn, rec, final_pos=fin[rec], ref_graph=rg, patch_end_layer=ell
        )
        for name, (sn, rg) in groups.items()
    }
    print(
        f"raw en->zh @ {s:g}x [ell={ell}]: "
        f"top {M.top_token_probs(res.ablated_logits[-1], tokenizer, k=3)}"
    )
    M.print_readout_row(f"  en->zh @ {s:g}x", row)
    ladder[f"{s:g}x"] = row
(OUT / "language_ladder_readouts_constrained.json").write_text(
    json.dumps(ladder, indent=1, ensure_ascii=False, default=str)
)

## F. Cross-lingual feature overlap by layer (Fig B7)

The paper's IOU protocol: per layer, the intersection-over-union of the feature sets
**active anywhere in the context**, on translated paragraphs, against an **unrelated-pair
baseline** (paragraph i in language A vs paragraph i+1 in language B). Paper shape: low at
both ends, mid-network peak well above baseline; EN-FR (shared alphabet) highest.

In [ ]:
corpus = M.CORPUS + M.PARAGRAPHS
curves = M.overlap_curves_paper(model, tc, tokenizer, corpus)
(OUT / "overlap_curves.json").write_text(
    json.dumps({k: v.tolist() for k, v in curves.items()}, indent=1)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for key, arr in curves.items():
    if key == "set-size":
        continue
    if key.endswith("baseline"):
        axes[0].plot(arr, "--", alpha=0.5, label=key)
    else:
        axes[0].plot(arr, lw=2.2 if key == "mean" else 1.3, label=key)
axes[0].set_xlabel("layer")
axes[0].set_ylabel("IOU of active-feature sets")
axes[0].set_title(f"Qwen3-{SIZE}: paper protocol + baseline")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)
for pair in ("en-fr", "en-zh", "fr-zh"):
    axes[1].plot(curves[pair] - curves[f"{pair}-baseline"], lw=1.6, label=pair)
axes[1].set_xlabel("layer")
axes[1].set_ylabel("IOU - baseline")
axes[1].set_title("baseline-subtracted (paper's quantity)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "overlap_paper.png", dpi=130)
plt.show()

lo, hi = N_LAYERS // 3, 2 * N_LAYERS // 3
for pair in ("en-fr", "en-zh", "fr-zh"):
    exc = curves[pair] - curves[f"{pair}-baseline"]
    print(
        f"{pair}: raw peak {curves[pair].max():.3f}@L{int(curves[pair].argmax())}, "
        f"baseline-subtracted mid-third mean {exc[lo:hi].mean():.3f}, ends {exc[0]:.3f}/{exc[-1]:.3f}"
    )
ss = curves["set-size"]
print(
    f"mean active-set size/layer: min {ss.min():.0f}, median {np.median(ss):.0f}, "
    f"max {ss.max():.0f} (dict 163840)"
)

## G. Which language is the mechanistic default?

Paper (§B.3.4, Claude-specific): multilingual say-large features push the **English**
token hardest — "English is mechanistically privileged as the default". The transferable
claim is that *some* language plays that role. We take the shared say-big features (late
layers, in ≥2 of the three antonym graphs among the influence-top answer features) and
compare each one's **direct decoder effect** on the EN/FR/ZH answer-token logits.

In [ ]:
tok_ids = {lg: behavior["antonym"][lg]["token_id"] for lg in M.LANGS}
member = {}
for lg in M.LANGS:
    for n in M.answer_features(graphs[f"antonym_{lg}"], top_k=12):
        member.setdefault((n["layer"], n["feature_idx"]), []).append(lg)
shared = {k: v for k, v in member.items() if len(v) >= 2 and k[0] >= N_LAYERS // 2}
effects = {}
for (L, f), langs_with in sorted(shared.items()):
    eff = M.direct_logit_effect(model, tc, L, f, tok_ids)
    effects[f"L{L}f{f}"] = eff
    print(
        f"L{L}f{f} (in {'/'.join(langs_with)}): "
        + ", ".join(f"{lg}={eff[lg]:+.3f}" for lg in M.LANGS)
    )
if effects:
    means = {lg: float(np.mean([e[lg] for e in effects.values()])) for lg in M.LANGS}
    print(
        f"\nmean direct effect over {len(effects)} shared say-big features: "
        + ", ".join(f"{lg}={means[lg]:.3f}" for lg in sorted(means, key=means.get, reverse=True))
    )

In [ ]:
# Explorer HTMLs with the REVIEWED supernodes pre-loaded as GROUPS (the review-page
# convention; "Export groups" round-trips them for any future re-review).
from llm_circuits.circuits.graph_explorer import render_graph_explorer_html

for name in GRAPH_NAMES:
    gspecs = []
    for sn_name, sn in SN.items():
        members = [
            (m["layer"], m["feature"]) for m in sn["members"] if name in (m.get("positions") or {})
        ]
        if members:
            gspecs.append({"name": sn_name, "members": members, "position": None})
    render_graph_explorer_html(graphs[name], OUT / f"graph_{name}.html", title=name, groups=gspecs)
    print(f"graph_{name}.html: {len(gspecs)} reviewed groups")

## H. Scale comparison: 4b vs 8b overlap (Fig B7's model comparison)

Paper: Haiku shares markedly more than its smaller 18L counterpart, most of all on the
non-alphabet-sharing pairs (EN-ZH, FR-ZH). This run uses the registry's **same-recipe
pair**: `mwhanna/qwen3-4b-transcoders` vs `mwhanna/qwen3-8b-transcoders` (both plain
recipe, both 36 layers, both 163,840 features/layer), with per-layer active-set sizes
printed as the granularity check (IOU is mechanically sensitive to how many features
fire). We free the 4b and run the identical overlap protocol on Qwen3-8B — overlap is
encode-only, so lazy decoders keep the 8B well inside an A100-80GB.

In [ ]:
del model, tc
gc.collect()
torch.cuda.empty_cache()

model8, tokenizer8 = load_qwen3("8b", dtype_str="bf16", device_map=DEVICE)
model8.eval()
tc8 = load_transcoder("qwen3-8b", device=DEVICE, dtype=torch.bfloat16).transcoder
curves8 = M.overlap_curves_paper(model8, tc8, tokenizer8, corpus)
OUT8 = artifacts_dir() / "paper_multilingual" / "8b"
OUT8.mkdir(parents=True, exist_ok=True)
(OUT8 / "overlap_curves.json").write_text(
    json.dumps({k: v.tolist() for k, v in curves8.items()}, indent=1)
)

# same-recipe granularity check (IOU is mechanically sensitive to active-set size)
for name, c in (("4b", curves), ("8b", curves8)):
    ss = c["set-size"]
    print(
        f"{name}: mean active-set size/layer min {ss.min():.0f} / median {np.median(ss):.0f} / "
        f"max {ss.max():.0f} (dict 163840)"
    )

fig, ax = plt.subplots(figsize=(7.5, 4.4))
for pair, color in (("en-fr", "C0"), ("en-zh", "C2"), ("fr-zh", "C4")):
    ax.plot(
        curves[pair] - curves[f"{pair}-baseline"], color=color, lw=1.4, ls="--", label=f"4b {pair}"
    )
    ax.plot(curves8[pair] - curves8[f"{pair}-baseline"], color=color, lw=1.8, label=f"8b {pair}")
ax.set_xlabel("layer (both models: 36)")
ax.set_ylabel("IOU - baseline")
ax.set_title("baseline-subtracted cross-lingual overlap: Qwen3-4B vs 8B (same recipe)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT8 / "overlap_scale_comparison.png", dpi=130)
plt.show()


def midmean(c, n, pair):
    exc = c[pair] - c[f"{pair}-baseline"]
    return float(exc[n // 3 : 2 * n // 3].mean())


for pair in ("en-fr", "en-zh", "fr-zh"):
    print(
        f"{pair}: baseline-subtracted mid-third mean  4b={midmean(curves, N_LAYERS, pair):.3f}  "
        f"8b={midmean(curves8, len(tc8), pair):.3f}"
    )

## Summary — verdicts vs the paper

_(filled from the executed run)_